# Building AI-Friendly Scientific Software: A Model Context Protocol Journey

In this post I'll share my recent (soon to be outdated) experience extending and maintaining a scientific codebase, [Napistu](https://github.com/napistu/napistu) using modern agentic code development (Cursor & Claude). 

Scraps:
- The goal is to put relevant information in an AIs hands on-demand.
- I'll discuss how I setup a Model Context Protocol (MCP) server and how I remotely deployed the server to Google Cloud run as part of my CI/CD.


## The AI development paradox

🚀 These approaches are amazing! They can massively increase productivity by efficiently tackling routine tasks, lowering the activation energy for skill development and simplifying debugging. These contributions not only save devs time, they also preserve our mental energy for critical tasks - I love just throwing a set of pytest failures at an agent and saying "you deal with it".

⚠️ But, this hands-off, vibe coding, mentality can easily get you into trouble. Its easy for AI agents to not follow common design patterns, to duplicate code, to over-engineer irrelevant features, and to produce straight-up brittle, buggy code. This can easily render a project unmaintainable.

For example, here's what happened when I asked Cursor to **add validation to the SBML parsing function** without any context about our existing patterns:

```python
def validate_sbml_file(file_path: str) -> bool:
    """Generic SBML validation using libsbml."""
    try:
        import libsbml
        document = libsbml.readSBML(file_path)
        if document.getNumErrors() > 0:
            return False
        return True
    except Exception:
        return False
```

The AI-generated code wasn't wrong, but it completely missed our existing SBML_dfs class and our domain-specific validation requirements. Multiply this across dozens of functions and you end up with a fragmented codebase.

🎯 Many of us are trying to navigate this tension - looking for a sweet spot where AI maximizes productivity. We are also looking for how to shift this sweet spot - finding tasks where agent performance can be shifted from dubious to quality.

**How do we get AI to understand our domain-specific codebase?**

Here, I'll share some approaches that I've found particularly helpful for scientific programming in the age of AI. A critical challenge is providing the right information up-front and allowing agents to surface relevant information on-demand. To do this we can use the Model Context Protocol (MCP) to provide information in a format suitable for AI models. In this post, I'll discuss the development of a Napistu MCP server, how I deployed it to Cloud Run via Github Actions, and provide some case studies of how agents operate with and without using the MCP server.

## Information is everything

**The core problem**: The core issue affects any domain-specific codebase - whether it's a financial trading system, game engine, or scientific library. AI agents lack the contextual knowledge to understand existing patterns, conventions, and domain-specific approaches.

Let's trace through how this plays out with a real example. Say I provide the following prompt:

> How do I create a consensus network from multiple pathway databases in Napistu? Please create a single artifact with your initial thoughts.

**Without any context**, Claude has no idea what I'm talking about and starts Googling:

> I'm not familiar with Napistu as a specific software tool or platform for pathway database analysis. ... Since I cannot locate specific documentation for Napistu, I'll create an artifact with general guidance on creating consensus networks from multiple pathway databases

The [resulting artifact](https://claude.ai/public/artifacts/adc42e81-b7c9-4fba-bd1c-b59c5294de05) actually hits on many of the challenges I faced when developing Napistu but doesn't discuss Napistu itself.

**With relevant code context**, by pointing Claude to relevent .py files on GitHub, the response improves:

> Looking at the Napistu codebase, I can see this is a systems biology toolkit for working with pathway models. Let me create a comprehensive guide on how to create a consensus network from multiple pathway databases.

The [resulting artifact](https://claude.ai/public/artifacts/5d098345-d881-4122-b37d-91832dcaa72f) highlights some of the key classes and functions and organizes the results in an orderly progression. But, the response feels disjointed like it pulled snippets of information from lots of places but didn't synthesize them well. Moreover, producing this response required me to provide specific .py files to Claude because only ~15% of the *napistu.py* codebase could fit into Claude's context window.

In contrast, **an expert's response** draws from multiple information sources:

> Start with the [merging_models_into_a_consensus](https://github.com/napistu/napistu/blob/main/tutorials/merging_models_into_a_consensus.ipynb) tutorial - it walks through exactly this workflow. Building consensus involves calling calling consensus.construct_consensus_model() with multiple SBML_dfs objects and a pathway index which organizes the objects metadata. This is currently being reworked in [Issue 189](https://github.com/napistu/napistu-py/issues/189) to remove the pathway index requirement. Finally, read the [consensus Napistu Wiki page](https://github.com/napistu/napistu/wiki/Consensus) so that you understand the key algorithm at a high-level. 

The difference is stark: the expert seamlessly integrates code patterns, tutorials, community discussions, and implementation examples. The challenge isn't just context window limits - it's **information fragmentation** (knowledge scattered across repositories, wikis, issues, tutorials) and **signal vs. noise** (finding the exact relevant pattern among thousands of functions).

### Solution preview: What if an AI could retrieve domain-specific information on demand?

Before I talk about how to provide agent-friendly information information, I'll show you the fruits of that effort for Napistu. First, I will install Napistu with MCP dependencies enabled:

```bash
pip install 'napistu[mcp]'
```

Now, I can configure the remote documentation server's url and port with `production_client_config`:

In [1]:
import logging
logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(__name__)


In [2]:
from IPython.display import HTML, display
from napistu.mcp.config import production_client_config
config = production_client_config()

display(HTML(f"Host: {config.host}<br>Port: {config.port}"))

Since this is a remote server, I can now start interacting with the server. I'll ask our consensus modeling question again and then reformat the AI-friendly JSON output as human-friendly tables.

In [3]:
import pandas as pd
from napistu.mcp.client import search_component

QUERY = "How do I create a consensus network from multiple pathway databases in Napistu?"
COMPONENTS = ["codebase", "documentation", "tutorials"]

# Returns actual Napistu function signatures, docs, and usage examples

for component in COMPONENTS:
    result = await search_component(component, QUERY, config=config)
    display(HTML(f"<h5>{component}</h2>"))
    display(pd.DataFrame(result["results"]))


,content,metadata,source,similarity_score
0,napistu.consensus.napistu.consensus.construct_...,{'name': 'napistu.consensus.construct_consensu...,functions: napistu.consensus.construct_consens...,0.591287
1,napistu.network.net_create.napistu.network.net...,{'name': 'napistu.network.net_create.create_na...,functions: napistu.network.net_create.create_n...,0.561030
2,napistu.consensus.napistu.consensus.build_cons...,{'name': 'napistu.consensus.build_consensus_id...,functions: napistu.consensus.build_consensus_i...,0.549506
3,napistu.network.net_create.napistu.network.net...,{'name': 'napistu.network.net_create.process_n...,functions: napistu.network.net_create.process_...,0.537183
4,napistu.consensus.napistu.consensus.reduce_to_...,{'name': 'napistu.consensus.reduce_to_consensu...,functions: napistu.consensus.reduce_to_consens...,0.528715


,content,metadata,source,similarity_score
0,## Tutorials\n\nThese tutorials are intended a...,"{'chunk': 4, 'is_chunked': True, 'name': 'napi...",readme: napistu (part 5),0.714167
1,# Napistu\n\nThe Napistu project is an approac...,"{'chunk': 0, 'is_chunked': True, 'name': 'napi...",readme: napistu (part 1),0.685483
2,- Represent a range of publicly-available data...,"{'chunk': 1, 'is_chunked': True, 'name': 'napi...",readme: napistu (part 2),0.580290
3,**Formats used in Napistu:** Reactome's result...,"{'chunk': 3, 'is_chunked': True, 'name': 'Data...",wiki: Data-Sources (part 4),0.574175
4,Napistu's molecular graphs let us answer biolo...,"{'chunk': 0, 'is_chunked': True, 'name': 'Expl...",wiki: Exploring-Molecular-Relationships-as-Net...,0.569216


,content,metadata,source,similarity_score
0,"---\ntitle: ""Tutorial - Merging Networks into ...","{'chunk': 0, 'is_chunked': True, 'name': 'merg...",tutorials: merging_models_into_a_consensus (pa...,0.607556
1,## Load an `sbml_dfs` pathway representation\n...,"{'chunk': 1, 'is_chunked': True, 'name': 'crea...",tutorials: creating_a_napistu_graph (part 2),0.591179
2,"---\ntitle: ""Tutorial - Working with Genome-Sc...","{'chunk': 0, 'is_chunked': True, 'name': 'work...",tutorials: working_with_genome_scale_networks ...,0.574234
3,INFO:napistu.consensus:Merging reactions ident...,"{'chunk': 22, 'is_chunked': True, 'name': 'dow...",tutorials: downloading_pathway_data (part 23),0.551939
4,INFO:napistu.consensus:Creating source table\n...,"{'chunk': 4, 'is_chunked': True, 'name': 'merg...",tutorials: merging_models_into_a_consensus (pa...,0.543965


A lot of the information that the expert provided is returned in this initial query. But, the goal is not to provide ALL of the relevant information in one go because this would innevitably include a lot of irrelevant info. Rather, we can use tools like `search_component` to give agents agency - putting information at the tips of their virtual fingertips. This allows agents to nimbly explore a problem drawing upon relevant resources on-demand. The result, instead of hallucinating generic solutions, agents can then discover actual patterns, find relevant tutorials, and understand our domain-specific approaches.

### Enter Model Context Protocol (MCP)

MCP provides a standardized way for AI models to access external information sources. Think of MCP as giving AI agents a research assistant who knows your project inside and out - someone who can instantly find relevant documentation, code examples, and implementation patterns specific to your domain.

<< TO DO - discuss `tools` and `resources` here >>


# Anatomy of the Napistu MCP Server

<< TO DO - discuss my goal in creating the Napistu MCP >>

- 1. helps users
- 2. encourages community development (hopefully!)
- 3. helps core devs and super users


## FastMCP Foundation

The Model Context Protocol provides a standard way for AI models to access external information. [FastMCP](https://github.com/jlowin/fastmcp) gives us a Flask-like Python implementation:

```python
from fastmcp import FastMCP

mcp = FastMCP("napistu-server")

@mcp.resource("napistu://health")
async def health_check():
    return {"status": "healthy", "components": [...]}

@mcp.tool()
async def search_documentation(query: str) -> dict:
    return {"results": [...]}
```

FastMCP handles the protocol details - we focus on exposing Napistu's knowledge.

## Components

Napistu uses a component-based architecture which provides separation of concerns (each component manages its own data), graceful degradation (failed components don't break others), and flexible deployment (enable only needed components). This also lets me create servers which are tailored to different use cases (e.g., a local server which can execute Napistu code or a remote docs server).

The current components are:
- Documentation: READMEs, wiki pages, GitHub issues/PRs via API  
- Codebase: API docs and function signatures from ReadTheDocs  
- Tutorials: Jupyter notebooks converted to searchable markdown  
- Execution: Working with a live Python environment (in development)
- Health: Server monitoring and component diagnostics

Each component follows a consistent pattern: load data, register endpoints, handle search: (TO DO - MORE DETAILS ON INITIALIZATION AND REGISTRATION)

```python
class DocumentationComponent(MCPComponent):
    async def initialize(self, semantic_search: SemanticSearch = None) -> bool:
        """Load READMEs, wiki pages, GitHub issues"""
        # Load external data and populate component state
        return success
    
    def register(self, mcp: FastMCP) -> None:
        """Register resources and tools with MCP server"""
        @mcp.tool()
        async def search_documentation(query: str):
            return self.state.semantic_search.search(query, "documentation")
```

{% include ai-aside.html content="
Its important to have detailed AI-first docstrings for MCP resources and tools. This information is available to most agents before they use the server's endpoints so its helpful to clarify when and when NOT to use the method. All caps and bold sections are a bit obnoxious for humans but these do effectively draw agent's attention. For example, here is part of the docstring for the `search_codebase` tool.

    **USE THIS WHEN:**
    - Looking for specific Napistu functions, classes, or modules
    - Finding API documentation for Napistu features
    
    **DO NOT USE FOR:**
    - General programming concepts not specific to Napistu
    - Documentation for other libraries or frameworks
" %}

## Smart Search: Semantic + Vector Embeddings

We can search content using exact keywords (e.g., "create_consensus") or semantic search (e.g., "How do I merge pathway data?"). Semantic search utilizes a `SemanticSearch` object which is shared acrosss components:

1. **Content Processing**: Load content, chunk long documents at natural boundaries
2. **Embedding Generation**: Convert chunks to 384-dim vectors using `all-MiniLM-L6-v2` sentence transformer
3. **Vector Storage**: Store in ChromaDB with metadata 
4. **Query Processing**: Embed user queries, find nearest neighbors via cosine similarity

```python
class SemanticSearch:
    def __init__(self, persist_directory: str = "./chroma_db"):
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.embedding_function = SentenceTransformerEmbeddingFunction(
            model_name="all-MiniLM-L6-v2"
        )
    
    def search(self, query: str, collection_name: str):
        # Convert query to vector, find similar content by cosine similarity
        return similarity_results_with_scores
```

{% include ai-aside.html content="
An early version of the server used keyword-based search to comb through all of the cached information. The results quality was massively improved when moving to vector-based search but this required me to implement a few tricky new features. To approach this problem, I worked with Claude to research different approaches balancing projected performance, against ease of implementation and maintainability. Since it was doing a good job, rather than switching to Cursor, I stayed with Claude for adding the actual semantic search functionality. This worked pretty well because I could feed the whole *napistu.mcp* subpackage into its context and there was already a lot of structure to the codebase. It tried to add unnecessary complexity in a few places (like maintaining component-level `Chroma` databases rather than centralized one) but overall this went quite smoothly and I was able to get this functionality up-and-running in a few hours. 

While I keep a pretty "tight leash" on agents when contributing to the scientific portions of the Napistu codebase, I've provided agents with more room to operate in developing the *napistu.mcp* subpackage. To do this I focus more on code review to check the AI's assumptions (e.g., "do we really need to assign global variables?"), and to suggest refactoring (e.g., "would creating a `ServerProfile` class simplify component configuration?"). After a session of implementing features in Claude, I've used it to update the [Napistu MCP server Wiki](https://github.com/napistu/napistu/wiki/Model-Context-Protocol-(MCP)-server) with some extra guidance (shorten 4-fold, remove this section). By maintaining this high-level document (which can also be accessed via MCP), we are essentially helping agents "save their place" for future development sessions.
" %}

## Client-Server Protocol

<< TO DO - either here or in the first section I should talk about the actual server.py logic >>

Here's what agents actually send and receive:

```python
# Agent request
await call_server_tool("search_tutorials", {
    "query": "consensus networks getting started", 
    "search_type": "semantic"
})

# MCP server response
{
  "results": [{
    "content": "# Merging Models into a Consensus\n\nThis tutorial shows...",
    "source": "tutorials: merging_models_into_a_consensus (part 1)",
    "similarity_score": 0.89
  }]
}
```

Agents get structured, searchable access to domain-specific knowledge - like having an expert who knows exactly where to find relevant information.

## From local to global: deployment story

### Local development

It's easy to setup a local MCP server that digests relevant documents and interacts with local agents:

```bash
# Install Napistu with MCP dependencies
pip install 'napistu[mcp]'

# Start full development server (all components)
python -m napistu.mcp server full

# Health check shows component loading
python -m napistu.mcp health --local
```

```output
🏥 Napistu MCP Server Health Check
========================================
Server URL: http://127.0.0.1:8765/mcp

Server Status: healthy

Components:
  ✅ documentation: healthy
  ✅ codebase: healthy  
  ✅ tutorials: healthy
  ✅ semantic_search: healthy
```

But this requires installing Napistu, maintaining a background process, and keeping it running - that's a lot to ask of users who just want to explore the project or collaborate on development.

### The always-up solution

Instead, I wanted an always-available service that I and others could easily use without any local setup. This meant deploying to the cloud with automatic updates whenever the codebase changes as part of my [GitHub Actions-based CI/CD workflows]([Github Actions strategy](https://github.com/napistu/napistu/wiki/GitHub-Actions-napistu%E2%80%90py))

Every new tagged version triggers deployment to Google Cloud Run:

```yaml
# Deploy workflow - simplified view
on:
  workflow_run:
    workflows: ["Release"]  # Auto-deploy after successful release
    types: [completed]
  schedule:
    - cron: '0 10 * * *'  # Daily content refresh at 2 AM PST

jobs:
  deploy:
    steps:
      - name: Deploy to Cloud Run
        run: |
          gcloud run deploy napistu-mcp-server \
            --image="us-west1-docker.pkg.dev/.../napistu-mcp-server:latest" \
            --cpu=1 --memory=2Gi \
            --set-env-vars="MCP_PROFILE=docs"
```

The production setup runs the "docs" profile (documentation + codebase + tutorials, no execution component) with 1 CPU and 2Gi memory, costing less than $1 per day. Content is refreshed whenever a new version of *napistu-py* is released and nightly to capture the latest documentation changes.

## The payoff

Now any AI tool can access the Napistu knowledge base instantly at https://napistu-mcp-server-844820030839.us-west1.run.app. Users don't need to install anything, run local processes, or handle maintenance - they can simply configure their AI tools to use the shared knowledge base. The service automatically updates with the latest documentation and code changes, while Cloud Run handles scaling, health checks, and automatic restarts for high availability.

```json
// Claude Desktop / Cursor configuration
{
  "mcpServers": {
    "napistu": {
      "command": "npx",
      "args": ["mcp-remote", "https://napistu-mcp-server-844820030839.us-west1.run.app/mcp/"]
    }
  }
}
```

The result: Napistu's entire knowledge base becomes instantly searchable by AI agents worldwide, dramatically lowering the barrier to contribution and collaboration.

# Case studies: AI agents in action

Lowering Activation Energy: 

**The Mission**: Making Napistu accessible to new users and collaborators

## Case Study 1: Learning with Claude

 "I'm new to Napistu, can you explain the SBML_dfs and NapistuGraph data structures to me?"


## Case Study 2: Building with Cursor - "Can you help me "**

**Without MCP**:
- Video/screenshots of Cursor suggesting generic validation patterns
- Misses established Napistu conventions and class structures
- Generic Python code that doesn't integrate well

**With MCP**:
- Screenshot/video of Cursor in action showing:
  - *Callout: Cursor searches codebase for existing patterns via MCP*
  - *Callout: Discovers SBML_dfs validation methods automatically*
  - *Callout: Follows established Napistu testing patterns*
- Shows generated code that properly uses Napistu classes and follows project conventions
- Demonstrates understanding of existing API patterns

**The Result**: From "intimidating research codebase" to "approachable, guided experience"


## The Bigger Picture: Scaling Scientific Software
- **What Worked**: Specific wins in onboarding, debugging, feature development
- **Community Building**: Lowering barriers increases contributor diversity
- **The Network Effect**: Better AI tools → more contributors → better software
- **Open Science**: Making research code truly accessible

## **VIII. Future Directions** *(~300 words)*
- **Execution Component**: Live Python environment integration (currently incomplete)
- **Multi-language Support**: Extending beyond Python to R components
- **Advanced RAG**: Better document chunking and retrieval strategies
- **Community Feedback**: Learning from early adopters

## Getting started: using and contributing to Napistu

### **For Users & Contributors**: Connect to the Napistu MCP server
```json
{
  "mcpServers": {
    "napistu": {
      "command": "npx",
      "args": ["mcp-remote", "https://napistu-mcp-server-844820030839.us-west1.run.app/mcp/"]
    }
  }
}
```

### **Try It Out**: 
- Configure Claude Desktop or Cursor with the MCP server
- Ask questions about Napistu functionality
- Start contributing to issues with AI assistance
- Join our community discussions

### **Call to Action**: 
**Help us build the future of network biology software!** The MCP server is just the beginning - we need contributors who can help extend Napistu's capabilities, improve documentation, and make systems biology research more accessible to everyone.

### **Story Arc:**
Problem (AI chaos) → Solution (MCP) → Implementation → Real Impact → Lessons → Community Building

